# 04. Recommendation Evaluation

This notebook compares recommendation versions using completion, saves, shares, replacements, candidate-pool size, and route diversity.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

WORKSPACE = Path.cwd()
if WORKSPACE.name == "notebooks":
    WORKSPACE = WORKSPACE.parent

sys.path.insert(0, str(WORKSPACE / "src"))

from venture_ds import (
    load_exports,
    load_optional_export,
    safe_rate,
    wilson_interval,
    ordered_session_funnel,
    add_sample_warning,
)

exports = load_exports(WORKSPACE / "data" / "raw")
events = exports["events"]
sessions = exports["sessions"]
feedback = exports["feedback"]
inventory = exports["inventory"]

exports["_files"]

,dataset,path,found
0,events,C:\Users\ppmal\Documents\Projects\venture-prod...,True
1,sessions,C:\Users\ppmal\Documents\Projects\venture-prod...,True
2,feedback,C:\Users\ppmal\Documents\Projects\venture-prod...,True
3,inventory,C:\Users\ppmal\Documents\Projects\venture-prod...,True


In [2]:
def property_series(frame, key):
    return frame["properties"].map(
        lambda value: value.get(key)
        if isinstance(value, dict) else None
    )

generated = events[events["event_name"].eq("outing_generated")].copy()

if generated.empty:
    print("No versioned outing generations found.")
else:
    for key in [
        "recommendationVersion",
        "planStatus",
        "candidateCount",
        "actualStopCount",
        "categoryDiversity",
        "neighbourhoodDiversity",
        "stopKindDiversity",
    ]:
        generated[key] = property_series(generated, key)

    display(generated.head())

,event_id,visitor_id,session_id,event_name,page_path,event_version,device_class,referrer_host,listing_id,listing_type,...,client_created_at,created_at,properties,recommendationVersion,planStatus,candidateCount,actualStopCount,categoryDiversity,neighbourhoodDiversity,stopKindDiversity
16,17,54b79b11-8090-4f58-bb33-e9d102148f99,17c90d50-3141-4460-9962-9e4b290a5e0b,outing_generated,/build-outing/results,2,NaN,NaN,NaN,NaN,...,NaT,2026-07-20 20:20:19.456036+00:00,"{'partySize': 1, 'planStatus': 'partial', 'alc...",route-score-v1,partial,2,1,1,1,1


In [3]:
if not generated.empty:
    saved_sessions = set(
        events.loc[events["event_name"].eq("outing_saved"), "session_id"]
    )
    shared_sessions = set(
        events.loc[events["event_name"].eq("outing_shared"), "session_id"]
    )
    replaced_sessions = set(
        events.loc[events["event_name"].eq("outing_stop_replaced"), "session_id"]
    )

    rows = []
    for version, group in generated.groupby("recommendationVersion", dropna=False):
        session_ids = set(group["session_id"])
        total = len(group)
        rows.append({
            "recommendation_version": version or "missing",
            "generated_plans": total,
            "sessions": len(session_ids),
            "completion_rate_pct": round(
                100 * group["planStatus"].eq("complete").mean(), 1
            ),
            "save_rate_pct": round(
                100 * safe_rate(len(session_ids & saved_sessions), len(session_ids)), 1
            ),
            "share_rate_pct": round(
                100 * safe_rate(len(session_ids & shared_sessions), len(session_ids)), 1
            ),
            "replacement_rate_pct": round(
                100 * safe_rate(len(session_ids & replaced_sessions), len(session_ids)), 1
            ),
            "average_candidates": pd.to_numeric(
                group["candidateCount"], errors="coerce"
            ).mean(),
            "average_category_diversity": pd.to_numeric(
                group["categoryDiversity"], errors="coerce"
            ).mean(),
            "average_neighbourhood_diversity": pd.to_numeric(
                group["neighbourhoodDiversity"], errors="coerce"
            ).mean(),
        })

    recommendation_summary = pd.DataFrame(rows)
    display(recommendation_summary)

,recommendation_version,generated_plans,sessions,completion_rate_pct,save_rate_pct,share_rate_pct,replacement_rate_pct,average_candidates,average_category_diversity,average_neighbourhood_diversity
0,route-score-v1,1,1,0.0,100.0,100.0,100.0,2.0,1.0,1.0


## Product meaning

Recommendation quality is multi-dimensional. A version that increases completion but collapses neighbourhood diversity or increases replacements may not be genuinely better.